# **WECC SettingsBuilder / `pg_to_switch` Validation Report**

**Purpose:** document the full debugging process used to answer running WECC-only YAMLs/resource groups through Melek's SettingsBuilder workflow and then validating them with `pg_to_switch.py`.

## Final result

A WECC-only `pg_to_switch.py` validation run completed successfully for:

```bash
--case-id p1 --year 2024 --myopic
```

Final successful command:

```bash
cd ~/repos/Switch-USA-PG-ReEDS-local

PYTHONPATH="$PWD/PowerGenome:$PYTHONPATH" PYTHONUNBUFFERED=1 python -u pg_to_switch.py pg/settings_wecc_test switch/in_wecc --case-id p1 --year 2024 --myopic \
  2>&1 | tee wecc_pg_to_switch_2024_p1.log
```

Successful output folder:

```text
~/repos/Switch-USA-PG-ReEDS-local/switch/in_wecc/2024/p1
```

Successful scenario file:

```text
~/repos/Switch-USA-PG-ReEDS-local/switch/in_wecc/scenarios_p1.txt
```

The final output folder contained **48 files**.

Important limitation: `my_case_s10x7 --year 2024` was **not** a valid case-year combination in `pg/extra_inputs/scenario_inputs.csv`. That case had rows for 2035 and 2050 only. The valid 2024 validation case used here was `p1`.


## 1. Local machine paths used

This section records the exact local folders used during the debug run.

### Original working/debug repo under Documents

```text
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS-melek-year-test
```

This repo contained the generated WECC settings and large/non-git data inputs, but later showed macOS/iCloud dataless file issues that caused imports and file reads to hang.

### Clean local clone used for the successful validation

```text
/Users/laurenvo/repos/Switch-USA-PG-ReEDS-local
```

This was the repo where `pg_to_switch.py` successfully ran.

### Downloaded WECC settings folder from Jenny/Melek files

```text
/Users/laurenvo/Downloads/powergenome_settings
/Users/laurenvo/Downloads/powergenome_settings/settings
/Users/laurenvo/Downloads/powergenome_settings/data
/Users/laurenvo/Downloads/powergenome_settings/extra_inputs
```

Important file from this folder:

```text
/Users/laurenvo/Downloads/powergenome_settings/data/network_costs_35r_western_nercr.csv
```

### Downloaded WECC resource groups folder

```text
/Users/laurenvo/Downloads/resource_groups_resource_groups
```

Files observed there included:

```text
onshorewind_group.json
onshorewind_lcoe_resource_groups.parquet
solar_group.json
solar_lcoe_resource_groups.parquet
```

### Generated WECC settings folder

```text
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg/settings_wecc_test
```

Later copied into the clean clone:

```text
/Users/laurenvo/repos/Switch-USA-PG-ReEDS-local/pg/settings_wecc_test
```

### Generated WECC resource groups folder

```text
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg/resource_groups_wecc_test/ReEDS-cpas-patched
```

Later copied into the clean clone:

```text
/Users/laurenvo/repos/Switch-USA-PG-ReEDS-local/pg/resource_groups_wecc_test/ReEDS-cpas-patched
```


## 2. SettingsBuilder notebook path setup

The SettingsBuilder notebook needed explicit paths to the downloaded WECC settings and resource groups.

The path setup used during this debug pass was:

```python
from pathlib import Path

REPO_ROOT = Path.home() / "Documents" / "Switch-USA-PG-ReEDS-melek-year-test"

GUI_SETTINGS_ROOT = Path.home() / "Downloads" / "powergenome_settings"
GUI_SETTINGS_DIR = GUI_SETTINGS_ROOT / "settings"
GUI_DATA_DIR = GUI_SETTINGS_ROOT / "data"
GUI_EXTRA_INPUTS_DIR = GUI_SETTINGS_ROOT / "extra_inputs"

GUI_RESOURCE_GROUPS_DIR = Path.home() / "Downloads" / "resource_groups_resource_groups"

REF_SETTINGS_DIR = REPO_ROOT / "pg" / "settings"

OUTPUT_SETTINGS_DIR = REPO_ROOT / "pg" / "settings_wecc_test"
OUTPUT_RESOURCE_GROUPS_DIR = REPO_ROOT / "pg" / "resource_groups_wecc_test" / "ReEDS-cpas-patched"
```

Important resource-group correction:

```python
gui_rg_dir = GUI_RESOURCE_GROUPS_DIR
```

This mattered because constructing the resource group path from `GUI_SETTINGS_DIR.parent` pointed to the wrong place.


In [ ]:
# This cell is for documentation/reproduction if run on Lauren's machine.
# It shows the intended path setup for the WECC SettingsBuilder workflow.

from pathlib import Path

REPO_ROOT = Path.home() / "Documents" / "Switch-USA-PG-ReEDS-melek-year-test"

GUI_SETTINGS_ROOT = Path.home() / "Downloads" / "powergenome_settings"
GUI_SETTINGS_DIR = GUI_SETTINGS_ROOT / "settings"
GUI_DATA_DIR = GUI_SETTINGS_ROOT / "data"
GUI_EXTRA_INPUTS_DIR = GUI_SETTINGS_ROOT / "extra_inputs"

GUI_RESOURCE_GROUPS_DIR = Path.home() / "Downloads" / "resource_groups_resource_groups"

REF_SETTINGS_DIR = REPO_ROOT / "pg" / "settings"

OUTPUT_SETTINGS_DIR = REPO_ROOT / "pg" / "settings_wecc_test"
OUTPUT_RESOURCE_GROUPS_DIR = REPO_ROOT / "pg" / "resource_groups_wecc_test" / "ReEDS-cpas-patched"

paths = {
    "REPO_ROOT": REPO_ROOT,
    "GUI_SETTINGS_ROOT": GUI_SETTINGS_ROOT,
    "GUI_SETTINGS_DIR": GUI_SETTINGS_DIR,
    "GUI_DATA_DIR": GUI_DATA_DIR,
    "GUI_EXTRA_INPUTS_DIR": GUI_EXTRA_INPUTS_DIR,
    "GUI_RESOURCE_GROUPS_DIR": GUI_RESOURCE_GROUPS_DIR,
    "REF_SETTINGS_DIR": REF_SETTINGS_DIR,
    "OUTPUT_SETTINGS_DIR": OUTPUT_SETTINGS_DIR,
    "OUTPUT_RESOURCE_GROUPS_DIR": OUTPUT_RESOURCE_GROUPS_DIR,
}

for name, path in paths.items():
    print(f"{name}: {path}  exists={path.exists()}")


## 3. Clean clone was needed because the Documents repo had dataless/iCloud issues

The original repo under `~/Documents` showed symptoms consistent with macOS/iCloud placeholder files:

- `pg_to_switch.py --help` hung.
- Some imports hung.
- Some files showed attributes like `compressed,dataless`.

A clean clone outside `Documents` fixed these source-code read/import problems.

Command used:

```bash
mkdir -p ~/repos
cd ~/repos
git clone --recursive https://github.com/melek2/Switch-USA-PG-ReEDS.git Switch-USA-PG-ReEDS-local
cd Switch-USA-PG-ReEDS-local
conda activate switch-pg-reeds
```

Validation that the clean clone worked:

```bash
PYTHONPATH="$PWD/PowerGenome:$PYTHONPATH" python -c "import powergenome.load_profiles; print('load_profiles import ok')"
PYTHONPATH="$PWD/PowerGenome:$PYTHONPATH" python pg_to_switch.py --help
```

Observed success:

```text
load_profiles import ok
```

`pg_to_switch.py --help` showed the expected CLI signature:

```text
Usage: pg_to_switch.py [OPTIONS] SETTINGS_FILE RESULTS_FOLDER

Options:
  --case-id TEXT
  --year INTEGER
  --myopic / --no-myopic
  --pg-unit-bug / --no-pg-unit-bug
  --case-index INTEGER
```


In [ ]:
%%bash
# Reproduction check: run from /Users/laurenvo/repos/Switch-USA-PG-ReEDS-local

pwd
echo

echo "Python:"
which python
python --version
echo

echo "PowerGenome import check:"
PYTHONPATH="$PWD/PowerGenome:$PYTHONPATH" python -c "import powergenome.load_profiles; print('load_profiles import ok')"

echo
echo "pg_to_switch.py CLI check:"
PYTHONPATH="$PWD/PowerGenome:$PYTHONPATH" python pg_to_switch.py --help | head -60


## 4. Files copied into the clean clone

The clean clone fixed the source-code problem, but it did not contain large/non-git data inputs. These had to be copied from the original debug repo under `Documents`.

### 4.1 Copy generated WECC settings and resource groups

```bash
cd ~/repos/Switch-USA-PG-ReEDS-local

cp -R ~/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg/settings_wecc_test pg/
cp -R ~/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg/resource_groups_wecc_test pg/ 2>/dev/null || true
```

### 4.2 Copy SQLite databases

`pg/settings_wecc_test/env.yml` referenced:

```yaml
PUDL_DB: 'pg_data/pudl.2025_08.sqlite'
PG_DB: 'pg_data/pg_misc_tables_efs_2025.3.sqlite'
```

Files found in the old repo:

```text
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/pudl.2025_08.sqlite
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/pg_misc_tables_efs_2025.3.sqlite
```

Copy commands:

```bash
cd ~/repos/Switch-USA-PG-ReEDS-local
mkdir -p pg_data

cp -av ~/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/pudl.2025_08.sqlite pg_data/
cp -av ~/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/pg_misc_tables_efs_2025.3.sqlite pg_data/
```

Verified sizes:

```text
pg_misc_tables_efs_2025.3.sqlite    6.8G
pudl.2025_08.sqlite                 490M
```


In [ ]:
%%bash
# Verify copied database files in the clean clone.

ls -lh pg_data/pudl.2025_08.sqlite pg_data/pg_misc_tables_efs_2025.3.sqlite


## 5. Resource profile parquet files copied into the active resource group folder

The resource group JSONs referenced REV profile parquet files that were not initially in the expected folder.

Initial failures included:

```text
OSError: Passed non-file path: pg/resource_groups_wecc_test/ReEDS-cpas-patched/offshorewind_rev_profiles_20240801.parquet
```

and then:

```text
OSError: Passed non-file path: pg/resource_groups_wecc_test/ReEDS-cpas-patched/onshorewind_rev_profiles_20240801.parquet
```

The missing files were found in:

```text
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/profiles
```

Copy command:

```bash
cd ~/repos/Switch-USA-PG-ReEDS-local

find ~/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/profiles \
  -maxdepth 1 \
  -name "*rev_profiles_20240801.parquet" \
  -exec cp -av {} pg/resource_groups_wecc_test/ReEDS-cpas-patched/ \;
```

Observed copied files:

```text
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/profiles/onshorewind_rev_profiles_20240801.parquet -> pg/resource_groups_wecc_test/ReEDS-cpas-patched/onshorewind_rev_profiles_20240801.parquet
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/profiles/solar_rev_profiles_20240801.parquet -> pg/resource_groups_wecc_test/ReEDS-cpas-patched/solar_rev_profiles_20240801.parquet
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/profiles/offshorewind_rev_profiles_20240801.parquet -> pg/resource_groups_wecc_test/ReEDS-cpas-patched/offshorewind_rev_profiles_20240801.parquet
```


In [ ]:
%%bash
# Verify REV profile parquet files exist where the resource group JSONs expect them.

ls -lh pg/resource_groups_wecc_test/ReEDS-cpas-patched/*rev_profiles_20240801.parquet


## 6. Scenario case-year issue

An initial run attempted:

```bash
--case-id my_case_s10x7 --year 2024
```

This failed because `my_case_s10x7` did not have a 2024 row in `pg/extra_inputs/scenario_inputs.csv`.

Inspection command:

```bash
cd ~/repos/Switch-USA-PG-ReEDS-local

python - <<'PY'
import pandas as pd
from pathlib import Path

p = Path("pg/extra_inputs/scenario_inputs.csv")
df = pd.read_csv(p)

print("Columns:")
print(df.columns.tolist())

print("\nRows for my_case_s10x7:")
print(df[df["case_id"].astype(str).eq("my_case_s10x7")].to_string(index=False))

print("\nAll 2024 cases:")
print(df[df["year"].eq(2024)][["case_id", "year"]].drop_duplicates().sort_values("case_id").to_string(index=False))
PY
```

Observed:

```text
Rows for my_case_s10x7:
      case_id  year   time_series load_growth_flexibility load_growth allow_retirement policies resource_limits fuel_price_forecast
my_case_s10x7  2035 my_case_s10x7                    firm        none              yes  current            none               hist5
my_case_s10x7  2050 my_case_s10x7                    firm        none              yes  current            none               hist5

All 2024 cases:
case_id  year
     p1  2024
  s10x5  2024
   s1x1  2024
  s20x1  2024
   s4x1  2024
   s4x5  2024
```

So the validation used:

```bash
--case-id p1 --year 2024 --myopic
```


In [ ]:
import pandas as pd
from pathlib import Path

p = Path("pg/extra_inputs/scenario_inputs.csv")
df = pd.read_csv(p)

print("Rows for my_case_s10x7:")
display(df[df["case_id"].astype(str).eq("my_case_s10x7")])

print("All 2024 cases:")
display(df[df["year"].eq(2024)][["case_id", "year"]].drop_duplicates().sort_values("case_id"))


## 7. `scenario_management.yml` issue

One generated `scenario_management.yml` was incomplete. It only had:

```yaml
settings_management:
  all_years:
    allow_retirement: ~
```

This caused:

```text
AttributeError: 'NoneType' object has no attribute 'get'
```

Inspection showed:

```text
settings_management keys: ['all_years']

KEY: 'all_years'
TYPE: <class 'dict'>
SUBKEYS: ['allow_retirement']
```

Fix used for validation:

```bash
cd ~/repos/Switch-USA-PG-ReEDS-local

cp pg/settings_wecc_test/scenario_management.yml pg/settings_wecc_test/scenario_management.yml.broken
cp pg/settings/scenario_management.yml pg/settings_wecc_test/scenario_management.yml
```

After replacement, inspection showed full scenario-management keys:

```text
settings_management keys: ['all_years', 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2035, 2040, 2045, 2050]
all_years subkeys: ['time_series', 'load_growth', 'load_growth_flexibility', 'allow_retirement', 'resource_limits', 'policies', 'fuel_price_forecast']
```

Notebook implication: the SettingsBuilder workflow should not partially write this file. It should either copy the full reference file or generate all required keys.


In [ ]:
# Inspect scenario_management.yml structure.

from pathlib import Path
import yaml

p = Path("pg/settings_wecc_test/scenario_management.yml")
d = yaml.safe_load(p.read_text()) or {}
sm = d.get("settings_management")

print("scenario_management path:", p)
print("settings_management type:", type(sm))
print("settings_management keys:", list(sm.keys()) if isinstance(sm, dict) else sm)

if isinstance(sm, dict) and "all_years" in sm:
    print("all_years subkeys:", list((sm.get("all_years") or {}).keys()))


## 8. `pg/extra_outputs` directory issue

The run reached generator clustering and failed with:

```text
OSError: Cannot save file into a non-existent directory: 'pg/extra_outputs'
```

The code was trying to write:

```text
pg/extra_outputs/existing_gen_units.csv
```

Fix:

```bash
cd ~/repos/Switch-USA-PG-ReEDS-local

mkdir -p pg/extra_outputs
mkdir -p switch/in_wecc
```

After this, the run progressed past existing generator clustering.


In [ ]:
%%bash
# Verify extra output and Switch output directories.

ls -ld pg/extra_outputs
ls -ld switch/in_wecc


## 9. Distributed generation data issue

The settings showed:

```text
pg/settings_wecc_test/distributed_gen.yml:8:distributed_gen_fn: nrel_reeds_distr_pv_2025.2.0.parquet
pg/settings_wecc_test/distributed_gen.yml:9:distributed_gen_scenario: mid_case
```

The run failed during distributed generation profile creation with:

```text
ValueError: max() arg is an empty sequence
```

Cause: PowerGenome looked for files matching `*pop_weight*` in the distributed generation input folder and found none.

Search in old repo found:

```text
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/efs_files_utc/ipm_state_pop_weight_20210517.parquet
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/efs_files_utc/ipm_state_pop_weight_20220329.csv
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/NREL-dist-gen/nrel_ba_unity_pop_weight.csv
```

Fix:

```bash
cd ~/repos/Switch-USA-PG-ReEDS-local

cp -av ~/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/NREL-dist-gen pg_data/
```

Then verify:

```bash
find pg_data/NREL-dist-gen \
  -iname "nrel_reeds_distr_pv_2025.2.0.parquet" \
  -o -iname "*pop_weight*"
```


In [ ]:
%%bash
# Verify distributed generation inputs.

grep -R "distributed_gen_fn\|distributed_gen" -n pg/settings_wecc_test | head -80

echo
find pg_data/NREL-dist-gen \
  -iname "nrel_reeds_distr_pv_2025.2.0.parquet" \
  -o -iname "*pop_weight*"


## 10. Flexible demand / load adjustment file issue

The run then reached flexible demand resources and failed with:

```text
FileNotFoundError: [Errno 2] No such file or directory:
'/Users/laurenvo/repos/Switch-USA-PG-ReEDS-local/pg/extra_inputs/load_adjustments.csv.zip'
```

Search command:

```bash
cd ~/repos/Switch-USA-PG-ReEDS-local

find ~/Documents ~/Downloads ~/repos -name "load_adjustments.csv.zip"
```

Observed:

```text
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS/pg/extra_inputs/load_adjustments.csv.zip
/Users/laurenvo/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg/extra_inputs/load_adjustments.csv.zip
```

Fix:

```bash
cd ~/repos/Switch-USA-PG-ReEDS-local

cp -av ~/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg/extra_inputs/load_adjustments.csv.zip \
  pg/extra_inputs/
```


In [ ]:
%%bash
# Verify load adjustment input.

ls -lh pg/extra_inputs/load_adjustments.csv.zip


## 11. Main WECC region bug: `regional_capacity_reserves` still used old `p*` regions

After the missing files were fixed, the run got much farther:

- compiled existing generators
- created distributed generation profiles
- finished existing generator clusters
- built new generation resources
- gathered load data
- extracted generator variability data

Then it failed in later table creation:

```text
KeyError: "The region p35 in 'regional_capacity_reserves', CapRes_1 is not a valid model region."
```

Inspection command:

```bash
cd ~/repos/Switch-USA-PG-ReEDS-local

grep -R "regional_capacity_reserves\|p35" -n pg/settings_wecc_test
```

Observed:

```text
pg/settings_wecc_test/model_definition.yml:152:regional_capacity_reserves:
pg/settings_wecc_test/model_definition.yml:189:    p35: 0.020
pg/settings_wecc_test/scenario_management.yml:189:            p35: 2.542271297498131
pg/settings_wecc_test/scenario_management.yml:324:            p35: 3.1838622167267996
```

The immediate blocker was `model_definition.yml`. The active WECC model regions were:

```text
['AZ1', 'AZ2', 'AZ3', 'AZ4', 'CA1', 'CA2', 'CA3', 'CA4', 'CO1', 'CO2', 'ID1', 'ID2', 'ID3', 'MT1', 'MT2', 'MT3', 'MT4', 'NM1', 'NV1', 'NV2', 'OR1', 'OR2', 'OR3', 'SD1', 'TX1', 'UT1', 'UT2', 'WA1', 'WA2', 'WA3', 'WA4', 'WY1', 'WY2', 'WY3', 'WY4']
```

Patch used:

```bash
python - <<'PY'
from pathlib import Path
import yaml

settings_dir = Path("pg/settings_wecc_test")
p = settings_dir / "model_definition.yml"

# Find model_regions from the WECC settings files
model_regions = None
for yml in settings_dir.glob("*.yml"):
    d = yaml.safe_load(yml.read_text()) or {}
    if "model_regions" in d:
        model_regions = d["model_regions"]
        print("Found model_regions in:", yml)
        break

if not model_regions:
    raise SystemExit("Could not find model_regions")

d = yaml.safe_load(p.read_text()) or {}

backup = p.with_suffix(".yml.bak_capres")
backup.write_text(p.read_text())

d["regional_capacity_reserves"] = {
    "CapRes_1": {r: 0.020 for r in model_regions}
}

p.write_text(yaml.safe_dump(d, sort_keys=False))

print("Patched:", p)
print("Backup:", backup)
print("Number of WECC model regions:", len(model_regions))
print(model_regions)
PY
```

Observed patch output:

```text
Found model_regions in: pg/settings_wecc_test/model_definition.yml
Patched: pg/settings_wecc_test/model_definition.yml
Backup: pg/settings_wecc_test/model_definition.yml.bak_capres
Number of WECC model regions: 35
['AZ1', 'AZ2', 'AZ3', 'AZ4', 'CA1', 'CA2', 'CA3', 'CA4', 'CO1', 'CO2', 'ID1', 'ID2', 'ID3', 'MT1', 'MT2', 'MT3', 'MT4', 'NM1', 'NV1', 'NV2', 'OR1', 'OR2', 'OR3', 'SD1', 'TX1', 'UT1', 'UT2', 'WA1', 'WA2', 'WA3', 'WA4', 'WY1', 'WY2', 'WY3', 'WY4']
```

Verification after patch:

```bash
grep -n "regional_capacity_reserves\|p35" pg/settings_wecc_test/model_definition.yml
```

Observed:

```text
114:regional_capacity_reserves:
```

`p35` was gone from `model_definition.yml`.


In [ ]:
# Inspect and optionally patch regional_capacity_reserves.
# This cell defaults to inspection only. Set APPLY_PATCH=True to rewrite model_definition.yml.

from pathlib import Path
import yaml

APPLY_PATCH = False

settings_dir = Path("pg/settings_wecc_test")
model_definition = settings_dir / "model_definition.yml"

model_regions = None
for yml in settings_dir.glob("*.yml"):
    d = yaml.safe_load(yml.read_text()) or {}
    if "model_regions" in d:
        model_regions = d["model_regions"]
        print("Found model_regions in:", yml)
        break

if not model_regions:
    raise SystemExit("Could not find model_regions")

d = yaml.safe_load(model_definition.read_text()) or {}

print("Current regional_capacity_reserves:")
print(d.get("regional_capacity_reserves"))

new_regional_capacity_reserves = {
    "CapRes_1": {r: 0.020 for r in model_regions}
}

print("\nProposed WECC regional_capacity_reserves:")
print(new_regional_capacity_reserves)

if APPLY_PATCH:
    backup = model_definition.with_suffix(".yml.bak_capres")
    backup.write_text(model_definition.read_text())
    d["regional_capacity_reserves"] = new_regional_capacity_reserves
    model_definition.write_text(yaml.safe_dump(d, sort_keys=False))
    print("\nPatched:", model_definition)
    print("Backup:", backup)
else:
    print("\nInspection only. Set APPLY_PATCH=True to patch.")


## 12. Final successful validation run

After fixing the `regional_capacity_reserves` issue, the same `pg_to_switch.py` command completed successfully:

```bash
cd ~/repos/Switch-USA-PG-ReEDS-local

PYTHONPATH="$PWD/PowerGenome:$PYTHONPATH" PYTHONUNBUFFERED=1 python -u pg_to_switch.py pg/settings_wecc_test switch/in_wecc --case-id p1 --year 2024 --myopic \
  2>&1 | tee wecc_pg_to_switch_2024_p1.log
```

The final log reached the clean-inputs post-processing step:

```text
================================================================================
Running 'Clean inputs' script:
/Users/laurenvo/miniforge3/envs/switch-pg-reeds/bin/python adjust/clean_inputs.py switch/in_wecc/2024/p1 --settings pg/settings_wecc_test
--------------------------------------------------------------------------------
```

Important final log lines:

```text
[enrich] wrote gen_info.csv (829/868 matched to existing units) and candidate_sites.csv (39 candidates).
[biopower-fuel] gen_info.csv: no matches.
[biopower-fuel] candidate_sites.csv: no matches.
[biopower-fuel] graph_tech_types.csv: no matches.
[biopower-hr] gen_info.csv: no '.' heat rates to fill.
[biopower-hr] candidate_sites.csv: no '.' heat rates to fill.
[ccs-load] gen_info.csv: added gen_ccs_energy_load column (0 CCS rows set to 0).
[ccs-load] candidate_sites.csv: added gen_ccs_energy_load column (0 CCS rows set to 0).
[geothermal] cap by zone (MW), 17 non-zero of 35:
zone
CA3    2839.7
NV2    1033.3
OR1     392.9
OR3     272.1
CA1     181.4
ID1     143.9
CA4     105.7
UT1     100.1
ID3      64.5
OR2      40.6
WY1      38.7
MT1      29.1
AZ1      26.3
WA1      22.7
NM1      21.6
CO2      12.2
CO1       8.3
[geothermal] gen_info: kept 0 / 0 geothermal rows, dropped 0 infeasible candidate(s).
[geothermal] candidate_sites: kept 0 / 0 geothermal rows, dropped 0 infeasible candidate(s).
[biomass] fuel_cost.csv: dropped 35 waste_biomass row(s); archived original to fuel_cost_archive.csv.
[biomass] wrote regional_fuel_markets.csv (35 rows), zone_to_regional_fuel_market.csv (35 rows), fuel_supply_curves.csv (233 rows = 35 markets x 1 periods).
[hydro-flow] MT4_conventional_hydroelectric_1: cap=198.00 MW, availability=1.000, ceiling=198.00 MW; max min_flow was 217.30 MW; clipped 4 value(s).
[hydro-flow] OR3_conventional_hydroelectric_1: cap=217.65 MW, availability=1.000, ceiling=217.65 MW; max min_flow was 220.03 MW; clipped 1 value(s).
[hydro-flow] clipped 5 hydro_min_flow_mw value(s) across 2 project(s); archived original to hydro_timeseries_archive.csv.
================================================================================

pg_to_switch:
created switch/in_wecc/scenarios_p1.txt
```

Error grep returned no output:

```bash
grep -n "Traceback\|Error\|FileNotFoundError\|KeyError\|ValueError\|OSError" wecc_pg_to_switch_2024_p1.log | tail -40
```

Output folder file count:

```bash
find switch/in_wecc/2024/p1 -maxdepth 1 -type f | wc -l
```

Observed:

```text
48
```


In [ ]:
%%bash
# Successful-run checks.
# Run from /Users/laurenvo/repos/Switch-USA-PG-ReEDS-local after the validation command.

echo "End of log:"
tail -n 80 wecc_pg_to_switch_2024_p1.log

echo
echo "Errors in log:"
grep -n "Traceback\|Error\|FileNotFoundError\|KeyError\|ValueError\|OSError" wecc_pg_to_switch_2024_p1.log | tail -40 || true

echo
echo "Output file count:"
find switch/in_wecc/2024/p1 -maxdepth 1 -type f | wc -l

echo
echo "Output files:"
ls -lh switch/in_wecc/2024/p1 | head -60

echo
echo "Scenario file:"
ls -lh switch/in_wecc/scenarios_p1.txt
cat switch/in_wecc/scenarios_p1.txt


## 13. Output files created in `switch/in_wecc/2024/p1`

The output folder contained 48 files. The first files listed were:

```text
candidate_sites_archive.csv
candidate_sites.csv
carbon_policies_regional.csv
carbon_policies.csv
existing_gen_units.csv
financials.csv
fuel_cost_archive.csv
fuel_cost.csv
fuel_supply_curves.csv
fuels.csv
gen_build_costs.csv
gen_build_predetermined.csv
gen_info_archive.csv
gen_info.csv
gen_om_by_period.csv
graph_tech_colors.csv
graph_tech_types.csv
graph_timestamp_map.csv
hydro_generation_projects.csv
hydro_timepoints.csv
hydro_timeseries_archive.csv
hydro_timeseries.csv
load_zones.csv
loads.csv
lost_load_cost.csv
max_cap_generators.csv
max_cap_requirements.csv
min_cap_generators.csv
min_cap_requirements.csv
```

Observed sizes from the terminal included:

```text
existing_gen_units.csv      1.1M
gen_build_costs.csv         104K
gen_build_predetermined.csv 83K
gen_info.csv                143K
loads.csv                   42M
hydro_timepoints.csv        1.2M
hydro_timeseries.csv        144K
```


In [ ]:
%%bash
# Show all output files and sizes.

ls -lh switch/in_wecc/2024/p1


## 14. Transmission-specific QA

Suspected the transmission section could be an issue.

Important WECC-specific transmission input:

```text
/Users/laurenvo/Downloads/powergenome_settings/data/network_costs_35r_western_nercr.csv
```

The notebook should explicitly check that the generated settings point to this file or its copied equivalent, and that transmission region endpoints match the active 35 WECC `model_regions`.

Use the following cells to inspect transmission-related settings and region labels.


In [ ]:
%%bash
# Search transmission-related settings.

echo "Transmission-related lines in pg/settings_wecc_test:"
grep -R "network_costs\|transmission\|tx_" -n pg/settings_wecc_test | head -160

echo
echo "Network cost files found under pg:"
find pg -name "network_costs_35r_western_nercr.csv" -o -name "*network_costs*western*" | sort


In [ ]:
# Structured check of network-cost CSV region-like columns against model_regions.
# This is intentionally conservative and meant for QA, not automatic failure.

from pathlib import Path
import pandas as pd
import yaml

settings_dir = Path("pg/settings_wecc_test")

model_regions = None
for yml in settings_dir.glob("*.yml"):
    d = yaml.safe_load(yml.read_text()) or {}
    if "model_regions" in d:
        model_regions = set(map(str, d["model_regions"]))
        break

print("model_regions count:", len(model_regions))
print(sorted(model_regions))

candidates = sorted(set(
    list(Path("pg").rglob("network_costs_35r_western_nercr.csv")) +
    list(Path("pg").rglob("*network_costs*western*")) +
    list(Path.home().joinpath("Downloads/powergenome_settings").rglob("network_costs_35r_western_nercr.csv"))
))

print("\nCandidate network cost files:")
for c in candidates:
    print(c)

for c in candidates:
    print("\n---", c, "---")
    try:
        df = pd.read_csv(c)
    except Exception as e:
        print("Could not read:", e)
        continue

    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    display(df.head())

    region_like_values = set()
    for col in df.columns:
        lower = col.lower()
        if any(token in lower for token in ["region", "zone", "from", "to"]):
            vals = df[col].dropna().astype(str).unique().tolist()
            print(f"{col}: sample={vals[:20]}")
            region_like_values.update(vals)

    outside = sorted(v for v in region_like_values if v not in model_regions)
    print("region-like values not in model_regions, sample:", outside[:50])


## 15. Non-WECC reference QA

The successful run still printed warnings about many unused full-system policy tags, such as `ESR_NY_rps`, `ESR_MA_ces`, `MinCapTag_NY_offshorewind`, etc.

Important distinction:

- **Fatal model-region mismatch:** old `p*` regions inside settings fields that must match `model_regions`. Example: `p35` in `regional_capacity_reserves`.
- **Non-fatal but noisy full-system leftovers:** unused ESR/model tags listed in `model_tags_name` but not assigned to resources.

The warning looked like:

```text
WARNING: The model resource tags {... 'ESR_NY_rps', 'ESR_MA_ces', ...} are listed
in the settings parameter 'model_tags_name' but are not assigned values for any resources
```

This warning did not stop the successful run, but it is relevant to avoiding mentions of other regions/states outside WECC. These tags should be filtered or cleaned if the generated WECC settings are expected to be free of non-WECC policy references.


In [ ]:
%%bash
# QA: search for p-region labels and non-WECC policy markers in generated settings.

echo "p-region labels in pg/settings_wecc_test:"
grep -R -nE '\bp[0-9]+\b' pg/settings_wecc_test | head -200 || true

echo
echo "Selected non-WECC policy markers in pg/settings_wecc_test:"
grep -R -nE 'ESR_(NY|MA|NJ|ME|RI|CT|MD|VA|NC|PA|OH|IL|MI|MN|MO|VT|NH)|MinCapTag_(NY|MA|NJ|ME|RI|CT|MD|VA)' pg/settings_wecc_test | head -200 || true


In [ ]:
# Structured YAML search for p-region keys.

from pathlib import Path
import yaml
import re

settings_dir = Path("pg/settings_wecc_test")
p_region_key = re.compile(r"^p\d+$")

def walk(obj, path=""):
    hits = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = str(k)
            new_path = f"{path}/{key}" if path else key
            if p_region_key.match(key):
                hits.append(new_path)
            hits.extend(walk(v, new_path))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hits.extend(walk(v, f"{path}[{i}]"))
    return hits

for yml in sorted(settings_dir.glob("*.yml")):
    d = yaml.safe_load(yml.read_text()) or {}
    hits = walk(d)
    if hits:
        print("\n", yml)
        for h in hits[:100]:
            print(" ", h)
        if len(hits) > 100:
            print("  ...", len(hits) - 100, "more")


## 16. Resource group QA

Resource group validation was another important part of the debugging process.

The resource group folder used in the clean clone was:

```text
/Users/laurenvo/repos/Switch-USA-PG-ReEDS-local/pg/resource_groups_wecc_test/ReEDS-cpas-patched
```

Files observed there included:

```text
offshorewind_fixed_false.json
offshorewind_fixed_true.json
offshorewind_floating_false.json
offshorewind_floating_true.json
offshorewind_lcoe_ReEDS_fixed_false.csv
offshorewind_lcoe_ReEDS_fixed_true.csv
offshorewind_lcoe_ReEDS_floating_false.csv
offshorewind_lcoe_ReEDS_floating_true.csv
offshorewind_lcoe_ReEDS.csv
onshorewind_group.json
onshorewind_lcoe_resource_groups.parquet
solar_group.json
solar_lcoe_resource_groups.parquet
```

Important concern: offshore wind files were present and generated repeated warnings like:

```text
WARNING: You have a renewables_cluster for technology 'offshorewind'
in region 'AZ1', but no comparable new-build technology was specified
in your settings file.
```

The warnings did not stop the final `p1` 2024 run, but offshore wind resource files should be reviewed if the goal is a clean WECC-only workflow.


In [ ]:
from pathlib import Path
import json

root = Path("pg/resource_groups_wecc_test")
json_files = sorted(root.rglob("*.json"))

print("Number of resource-group JSON files:", len(json_files))
for path in json_files:
    print(path)

missing_refs = []

for path in json_files:
    try:
        data = json.loads(path.read_text())
    except Exception as e:
        print("Could not load", path, e)
        continue

    for key in ["metadata", "profiles", "site_map"]:
        val = data.get(key)
        if isinstance(val, str):
            ref = path.parent / val
            if not ref.exists():
                missing_refs.append((str(path), key, val, str(ref)))

print("\nMissing JSON file references:", len(missing_refs))
for item in missing_refs:
    print(item)


In [ ]:
%%bash
echo "Resource group files:"
find pg/resource_groups_wecc_test -maxdepth 3 -type f | sort | sed -n '1,220p'

echo
echo "Offshore-related files:"
find pg/resource_groups_wecc_test -maxdepth 3 -type f \( -iname "*offshore*" -o -iname "*osw*" \) | sort


## 17. Summary table of failures and fixes

| Order | Failure / symptom | Cause | Fix |
|---:|---|---|---|
| 1 | `pg_to_switch.py --help` or PowerGenome imports hung | Original repo under `Documents` had macOS/iCloud dataless placeholder issues | Clean clone under `/Users/laurenvo/repos/Switch-USA-PG-ReEDS-local` |
| 2 | `OperationalError: unable to open database file` | Clean clone missing SQLite DBs referenced by `env.yml` | Copied `pudl.2025_08.sqlite` and `pg_misc_tables_efs_2025.3.sqlite` into `pg_data/` |
| 3 | `my_case_s10x7 --year 2024` failed case selection | `my_case_s10x7` only had 2035 and 2050 rows | Used valid 2024 case `p1` |
| 4 | `AttributeError: 'NoneType' object has no attribute 'get'` | `scenario_management.yml` was incomplete | Replaced with full reference `pg/settings/scenario_management.yml` |
| 5 | Missing offshore REV profile parquet | Resource-group JSON referenced file not copied into active folder | Copied `offshorewind_rev_profiles_20240801.parquet` |
| 6 | Missing onshore wind REV profile parquet | Same resource-group file-copy issue | Copied all `*rev_profiles_20240801.parquet` files |
| 7 | `Cannot save file into a non-existent directory: 'pg/extra_outputs'` | Extra outputs folder missing | Created `pg/extra_outputs` |
| 8 | `ValueError: max() arg is an empty sequence` | Distributed-gen `*pop_weight*` file missing | Copied `pg_data/NREL-dist-gen` |
| 9 | Missing `load_adjustments.csv.zip` | Flexible demand input missing from clean clone | Copied file into `pg/extra_inputs/` |
| 10 | `p35` not a valid model region | `regional_capacity_reserves` still used old `p*` regions | Patched `model_definition.yml` to use 35 WECC regions |
| 11 | Final run completed | All required inputs/settings issues fixed enough for validation | Output written to `switch/in_wecc/2024/p1` |


## 18. Recommended changes to Melek's SettingsBuilder workflow

Based on this debug pass, the notebook should be updated to:

1. Parameterize WECC input paths:
   - `GUI_SETTINGS_DIR`
   - `GUI_DATA_DIR`
   - `GUI_EXTRA_INPUTS_DIR`
   - `GUI_RESOURCE_GROUPS_DIR`
   - `OUTPUT_SETTINGS_DIR`
   - `OUTPUT_RESOURCE_GROUPS_DIR`

2. Explicitly use the WECC transmission file:
   - `network_costs_35r_western_nercr.csv`

3. Add a transmission validation cell:
   - print active transmission file
   - inspect endpoints/regions
   - compare endpoints to the 35 active `model_regions`

4. Add a resource group validation cell:
   - load all resource-group JSONs
   - verify every referenced `metadata`, `profiles`, and `site_map` file exists

5. Ensure `scenario_management.yml` is complete:
   - do not partially write only `allow_retirement`
   - either copy full reference file or generate all required keys

6. Generate `regional_capacity_reserves` from active `model_regions`:
   - do not inherit old `p*` regions like `p35`

7. Add non-WECC reference QA:
   - search generated settings for `p*` region labels
   - search for policy tags from states outside WECC
   - distinguish fatal region mismatches from harmless unused warning tags

8. Review offshore wind resource groups:
   - decide whether offshore wind should be included in this WECC test
   - if included, filter to appropriate WECC coastal regions
   - if not included, omit offshore JSONs from the active resource group folder


## 19. Update

Able to get a WECC-only `pg_to_switch` validation run working for the 2024 `p1` case using the settings generated from Melek's SettingsBuilder workflow.

The successful output folder is:

```text
~/repos/Switch-USA-PG-ReEDS-local/switch/in_wecc/2024/p1
```

The run created 48 output files and completed through the `clean_inputs` post-processing step, including creating:

```text
switch/in_wecc/scenarios_p1.txt
```

Main issues I had to resolve:

- The original repo under `Documents` had macOS/iCloud dataless placeholder issues, so PowerGenome imports and file reads were hanging. I moved to a clean local clone under `~/repos`.
- The clean clone needed non-git `pg_data` inputs copied over, including the PUDL/PG SQLite databases, REV profile parquet files, NREL distributed-gen inputs, and `load_adjustments.csv.zip`.
- `my_case_s10x7 --year 2024` was not a valid scenario combination because that case only has 2035 and 2050 rows, so I used `p1 --year 2024` for validation.
- The generated `scenario_management.yml` was incomplete in one pass, so I replaced it with the full reference `scenario_management.yml` for validation.
- `regional_capacity_reserves` in `model_definition.yml` still used old `p*` region labels like `p35`, which caused a hard failure because the WECC model regions are `AZ1`, `CA1`, etc. I patched that setting to use the 35 WECC model regions.
- The transmission input should be checked explicitly to confirm that the WECC-specific file `network_costs_35r_western_nercr.csv` is being used.

The run still produces warnings about unused ESR/model tags and renewable clusters with no comparable new-build technology, especially around offshore wind. Those are likely the next cleanup items if we want the generated WECC settings to contain no non-WECC policy/resource references. But the WECC `p1` 2024 `pg_to_switch` validation itself now completes successfully.


## 20. Exact command sequence from the successful debug pass

```bash
# Clean clone
mkdir -p ~/repos
cd ~/repos
git clone --recursive https://github.com/melek2/Switch-USA-PG-ReEDS.git Switch-USA-PG-ReEDS-local
cd Switch-USA-PG-ReEDS-local
conda activate switch-pg-reeds

# Confirm clean clone imports/CLI
PYTHONPATH="$PWD/PowerGenome:$PYTHONPATH" python -c "import powergenome.load_profiles; print('load_profiles import ok')"
PYTHONPATH="$PWD/PowerGenome:$PYTHONPATH" python pg_to_switch.py --help

# Copy generated WECC settings/resource groups
cp -R ~/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg/settings_wecc_test pg/
cp -R ~/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg/resource_groups_wecc_test pg/ 2>/dev/null || true

# Copy SQLite DBs
mkdir -p pg_data
cp -av ~/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/pudl.2025_08.sqlite pg_data/
cp -av ~/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/pg_misc_tables_efs_2025.3.sqlite pg_data/

# Replace incomplete scenario management
cp pg/settings_wecc_test/scenario_management.yml pg/settings_wecc_test/scenario_management.yml.broken
cp pg/settings/scenario_management.yml pg/settings_wecc_test/scenario_management.yml

# Copy REV profile parquets
find ~/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/profiles \
  -maxdepth 1 \
  -name "*rev_profiles_20240801.parquet" \
  -exec cp -av {} pg/resource_groups_wecc_test/ReEDS-cpas-patched/ \;

# Create required output folders
mkdir -p pg/extra_outputs
mkdir -p switch/in_wecc

# Copy distributed generation inputs
cp -av ~/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg_data/NREL-dist-gen pg_data/

# Copy flexible demand input
cp -av ~/Documents/Switch-USA-PG-ReEDS-melek-year-test/pg/extra_inputs/load_adjustments.csv.zip \
  pg/extra_inputs/

# Patch regional_capacity_reserves
python - <<'PY'
from pathlib import Path
import yaml

settings_dir = Path("pg/settings_wecc_test")
p = settings_dir / "model_definition.yml"

model_regions = None
for yml in settings_dir.glob("*.yml"):
    d = yaml.safe_load(yml.read_text()) or {}
    if "model_regions" in d:
        model_regions = d["model_regions"]
        print("Found model_regions in:", yml)
        break

if not model_regions:
    raise SystemExit("Could not find model_regions")

d = yaml.safe_load(p.read_text()) or {}

backup = p.with_suffix(".yml.bak_capres")
backup.write_text(p.read_text())

d["regional_capacity_reserves"] = {
    "CapRes_1": {r: 0.020 for r in model_regions}
}

p.write_text(yaml.safe_dump(d, sort_keys=False))

print("Patched:", p)
print("Backup:", backup)
print("Number of WECC model regions:", len(model_regions))
print(model_regions)
PY

# Run validation
PYTHONPATH="$PWD/PowerGenome:$PYTHONPATH" PYTHONUNBUFFERED=1 python -u pg_to_switch.py pg/settings_wecc_test switch/in_wecc --case-id p1 --year 2024 --myopic \
  2>&1 | tee wecc_pg_to_switch_2024_p1.log

# Confirm success
tail -n 80 wecc_pg_to_switch_2024_p1.log
grep -n "Traceback\|Error\|FileNotFoundError\|KeyError\|ValueError\|OSError" wecc_pg_to_switch_2024_p1.log | tail -40
find switch/in_wecc/2024/p1 -maxdepth 1 -type f | wc -l
ls -lh switch/in_wecc/2024/p1 | head -30
```
